# LoRA 저랭크 미세조정 실습 - 추가 실습 코드: LoRA 레이어 구현

- Tutorial ID: `mod-lora-lab`
- Tutorial: LoRA 저랭크 미세조정 실습
- Section ID: `practice-lora-basic`
- Section: 추가 실습 코드: LoRA 레이어 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: LoRA 레이어 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 큰 가중치 W를 직접 바꾸지 않고 저랭크 A/B 업데이트가 어떻게 더해지는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

class LoRALayer:
    """
    LoRA: Low-Rank Adaptation
    
    원래 가중치 W에 low-rank 분해 BA를 더합니다:
    h = Wx + (BA)x = Wx + B(Ax)
    
    여기서:
    - A: (rank, in_features) - 다운 프로젝션
    - B: (out_features, rank) - 업 프로젝션
    - rank << min(in_features, out_features)
    """
    
        def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # 원본 가중치 (frozen)
        self.W = np.random.randn(out_features, in_features) * 0.02
        
        # LoRA 가중치 (trainable)
        # A: Gaussian 초기화
        self.A = np.random.randn(rank, in_features) * 0.01
        # B: Zero 초기화 -> 학습 초기 ΔW = 0
        self.B = np.zeros((out_features, rank))
        
        def forward(self, x):
        """
        Forward pass:
        h = Wx + scaling * B(Ax)
        """
        # 원본 경로 (frozen)
        h_original = np.matmul(x, self.W.T)
        
        # LoRA 경로 (trainable)
        h_lora = np.matmul(x, self.A.T)  # x @ A^T
        h_lora = np.matmul(h_lora, self.B.T)  # (x @ A^T) @ B^T
        h_lora = self.scaling * h_lora
        
        return h_original + h_lora
    
        def count_params(self):
        """파라미터 수 비교"""
        original = self.out_features * self.in_features
        lora = self.rank * self.in_features + self.out_features * self.rank
        return original, lora

# 테스트
np.random.seed(42)

# GPT-2 크기 예시
in_features = 768
out_features = 768
ranks = [1, 4, 8, 16, 32]

print("=" * 60)
print("LoRA: Low-Rank Adaptation 파라미터 효율성")
print("=" * 60)
print(f"\\n원본 레이어: {in_features} → {out_features}")
print(f"원본 파라미터: {in_features * out_features:,} ({in_features * out_features * 4 / 1e6:.2f} MB)")

print("\\n━━━ Rank에 따른 LoRA 파라미터 ━━━")
for rank in ranks:
    lora = LoRALayer(in_features, out_features, rank=rank)
    orig, lora_params = lora.count_params()
    ratio = lora_params / orig * 100
    
    print(f"  Rank {rank:2d}: {lora_params:6,} params ({ratio:.2f}%)")

# Forward pass 테스트
print("\\n━━━ Forward Pass 테스트 (Rank=4) ━━━")
lora_layer = LoRALayer(in_features, out_features, rank=4)

# 배치 입력
batch_size = 2
x = np.random.randn(batch_size, in_features)

output = lora_layer.forward(x)
print(f"입력: {x.shape}")
print(f"출력: {output.shape}")

# 학습 전: B=0이므로 LoRA 기여 = 0
print(f"\\n초기 상태 (B=0):")
print(f"  LoRA 기여: {np.abs(lora_layer.scaling * np.matmul(np.matmul(x, lora_layer.A.T), lora_layer.B.T)).max():.6f}")
print(f"  → 처음에는 원본 모델과 동일!")

# B에 값이 생기면 LoRA가 기여
lora_layer.B = np.random.randn(out_features, 4) * 0.1
print(f"\\n학습 후 (B≠0):")
print(f"  LoRA 기여 max: {np.abs(lora_layer.scaling * np.matmul(np.matmul(x, lora_layer.A.T), lora_layer.B.T)).max():.4f}")

print("\\n✅ LoRA: 0.5% 미만의 파라미터로 Fine-tuning!")